# Quasi-Normal Mode Ringdown: Boundary Dependence

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gpartin/LFMPublicExperiments/blob/main/notebooks/LFM_QNM_Ringdown.ipynb)

## What This Notebook Demonstrates

QNM damping depends on **boundary conditions**, not the source:

- **Absorbing boundaries**: energy escapes $\Rightarrow$ oscillations damp (like GR at $r \to \infty$)
- **Reflecting boundaries**: energy trapped $\Rightarrow$ oscillations persist (LFM: $\chi \to 0$ horizon)

This is key to LFM's prediction of **high Q-factor ringdowns**.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

chi0, kappa, c = 19.0, 0.016, 1.0
N, L = 512, 100.0
dx = L / N
dt = 0.3 * dx / c
x = np.linspace(0, L, N)
center = L / 2
absorb_width = 15.0

def run_qnm(boundary='absorbing', n_equil=2000, n_evolve=10000):
    # Initial source
    E_source = 10.0 * np.exp(-((x - center) / 5.0)**2)
    E_sq = E_source**2
    chi = np.sqrt(np.maximum(chi0**2 - kappa * E_sq, 0.01))
    E = np.zeros(N)
    E_prev = np.zeros(N)
    # Equilibrate
    for _ in range(n_equil):
        lap = np.zeros(N)
        lap[1:-1] = (E[:-2] + E[2:] - 2*E[1:-1]) / dx**2
        E_new = 2*E - E_prev + dt**2 * (c**2 * lap - chi**2 * E)
        E_new[0] = E_new[-1] = 0
        E_prev, E = E.copy(), E_new.copy()
    # Perturb: reduce source by 20%
    E_source_new = 8.0 * np.exp(-((x - center) / 5.0)**2)
    chi = np.sqrt(np.maximum(chi0**2 - kappa * E_source_new**2, 0.01))
    monitor = []
    mon_idx = N // 4
    for step in range(n_evolve):
        lap = np.zeros(N)
        lap[1:-1] = (E[:-2] + E[2:] - 2*E[1:-1]) / dx**2
        E_new = 2*E - E_prev + dt**2 * (c**2 * lap - chi**2 * E)
        if boundary == 'absorbing':
            # Absorbing sponge layer
            damping = np.ones(N)
            damping[:int(absorb_width/dx)] = np.linspace(0.9, 1.0, int(absorb_width/dx))
            damping[-int(absorb_width/dx):] = np.linspace(1.0, 0.9, int(absorb_width/dx))
            E_new *= damping
            E_new[0] = E_new[-1] = 0
        else:
            # Reflecting (Neumann): dE/dx = 0 at boundaries
            E_new[0] = E_new[1]
            E_new[-1] = E_new[-2]
        E_prev, E = E.copy(), E_new.copy()
        monitor.append(E[mon_idx])
    return np.array(monitor)

print('Running absorbing boundary...')
signal_abs = run_qnm('absorbing')
print('Running reflecting boundary...')
signal_ref = run_qnm('reflecting')

# Compute envelopes
from scipy.signal import hilbert
env_abs = np.abs(hilbert(signal_abs - np.mean(signal_abs)))
env_ref = np.abs(hilbert(signal_ref - np.mean(signal_ref)))

# Fit decay rates
t_arr = np.arange(len(signal_abs)) * dt
mid = len(t_arr) // 4
end = 3 * len(t_arr) // 4

log_env_abs = np.log(np.maximum(env_abs[mid:end], 1e-20))
log_env_ref = np.log(np.maximum(env_ref[mid:end], 1e-20))
gamma_abs = -np.polyfit(t_arr[mid:end], log_env_abs, 1)[0]
gamma_ref = -np.polyfit(t_arr[mid:end], log_env_ref, 1)[0]

print(f'\nAbsorbing: decay rate gamma = {gamma_abs:.4f}')
print(f'Reflecting: decay rate gamma = {gamma_ref:.4f}')
print(f'\nAbsorbing damped: {gamma_abs > 0.001}')
print(f'Reflecting undamped: {gamma_ref < 0.001}')
print(f'H0 REJECTED: {gamma_abs > 0.001 and gamma_ref < 0.001}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

ax = axes[0]
ax.plot(t_arr, signal_abs, 'b-', lw=0.5, alpha=0.5)
ax.plot(t_arr, env_abs, 'r-', lw=2, label=f'Envelope (gamma={gamma_abs:.4f})')
ax.set_title('Absorbing Boundaries: Energy Escapes -> Damping')
ax.set_xlabel('Time')
ax.set_ylabel('E(monitor)')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(t_arr, signal_ref, 'b-', lw=0.5, alpha=0.5)
ax.plot(t_arr, env_ref, 'r-', lw=2, label=f'Envelope (gamma={gamma_ref:.4f})')
ax.set_title('Reflecting Boundaries: Energy Trapped -> No Damping')
ax.set_xlabel('Time')
ax.set_ylabel('E(monitor)')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('QNM Ringdown: Boundary Conditions Determine Damping', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Key Result

- With **absorbing** boundaries, QNM oscillations damp (energy radiates away)
- With **reflecting** boundaries, oscillations persist (energy is trapped)
- In LFM, the cosmic edge ($\chi \to 0$) acts as a reflecting boundary
- This predicts **high Q-factor** ringdowns — a testable deviation from GR